# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [3]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
# Let's use Ollama:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")
MODEL="llama3.2:1b"

In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
* You are provided with a list of links found on a webpage.
* You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Services/Products pages, or Blog/News pages, or Careers/Jobs pages.
* The pages cold be on other languages special portuguese, or spanish
* You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "Jobs page", "url": "https://another.full.url/careers"},
        {"type": "Blog page", "url": "https://another.full.url/blog"},
        {"type": "Products page", "url": "https://another.full.url/products"},
        {"type": "Services page", "url": "https://another.full.url/services"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [13]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [18]:
def select_relevant_links(url):
    print("Started...")
    links = get_links_user_prompt(url)
    print("All links:\n", links)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": links}
        ],
        response_format={"type": "json_object"}
    )
    print("Let's see...")
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [20]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about me and about nebula page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'connect four game page',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'outsmart page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'news and press release pages',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'}]}

In [19]:
select_relevant_links("http://vitarts.com.br")

Started...
All links:
 
Here is the list of links on the website http://vitarts.com.br -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#content
http://vitarts.com.br/
http://vitarts.com.br/
http://vitarts.com.br/sobre-nos/
http://vitarts.com.br/servicos/
http://vitarts.com.br/dashboards/
http://vitarts.com.br/blog/
http://vitarts.com.br/contato/
#
#
https://www.linkedin.com/company-beta/16242353
https://www.facebook.com/vitars.com.br/
mailto:contato@vitarts.com.br
http://vitarts.com.br/author/vitra/
#
#
#
#
https://www.linkedin.com/company-beta/16242353
https://www.facebook.com/vitars.com.br/
#
https://www.monsterinsights.com/?utm_source=verifiedBadge&utm_medium=verifiedBadge&utm_campaign=verifiedbyMonsterInsights
#
Let's see...


{'links': [{'type': 'About page', 'url': 'http://vitarts.com.br/sobre-nos/'},
  {'type': 'Services/Products pages',
   'url': 'https://vitarts.com.br/servicos'}]}

In [7]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    print("-"*60)
    print("link_system_prompt:\n", link_system_prompt)
    print("-"*60)
    links = get_links_user_prompt(url)
    print("All links:\n", links)
    print("-"*60)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": links}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    relevant_links = json.loads(result)
    
    links = []
    print(relevant_links)
    for link in relevant_links['links']:
        print(link)
        if (link['url'] > '' ):
            links.append(link)

    print(f"Found {len(links)} relevant links")
    print("-"*60)
    return links

In [21]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:1b
------------------------------------------------------------
link_system_prompt:
 
* You are provided with a list of links found on a webpage.
* You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Services/Products pages, or Blog/News pages, or Careers/Jobs pages.
* The pages cold be on other languages special portuguese, or spanish
* You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "Jobs page", "url": "https://another.full.url/careers"},
        {"type": "Blog page", "url": "https://another.full.url/blog"},
        {"type": "Products page", "url": "https://another.full.url/products"},
        {"type": "Services page", "url": "https://another.full.url/services"}
    ]
}

------------------------

[{'type': 'About page',
  'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
 {'type': 'Posts', 'url': 'https://edwarddonner.com/posts/'},
 {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
 {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
 {'type': 'Facebook profile',
  'url': 'https://www.facebook.com/edward.donner.52'}]

In [22]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
------------------------------------------------------------
link_system_prompt:
 
* You are provided with a list of links found on a webpage.
* You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Services/Products pages, or Blog/News pages, or Careers/Jobs pages.
* The pages cold be on other languages special portuguese, or spanish
* You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "Jobs page", "url": "https://another.full.url/careers"},
        {"type": "Blog page", "url": "https://another.full.url/blog"},
        {"type": "Products page", "url": "https://another.full.url/products"},
        {"type": "Services page", "url": "https://another.full.url/services"}
    ]
}

--------------------------

[{'type': 'About page', 'url': 'https://endpoints.huggingface.co'},
 {'type': 'Brand', 'url': 'https://brand.huggingface.co'},
 {'type': 'Docs/transformers',
  'url': 'https://docs.huggingface.co/transformers'},
 {'type': 'docs/docs', 'url': 'https://docs.huggingface.co'},
 {'type': 'docs/blog', 'url': 'https://discuss.huggingface.co'},
 {'type': 'docs/docs', 'url': 'https://docs.huggingface.co'},
 {'type': 'docs/accelerate', 'url': 'https://docs.huggingface.co/accelerate'},
 {'type': 'changelog', 'url': 'https://changelog.huggingface.co'}]

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [ ]:
def fetch_page_and_all_relevant_links(url):
    """
    Fetch the content of a landing page and a set of relevant linked pages.

    This function performs a two-step crawl:
    1. Retrieves the main (landing) page content.
    2. Identifies a set of relevant links from that page and fetches their content.

    The results are combined into a single Markdown-formatted string, where:
    - The landing page content appears under "## Landing Page"
    - Each relevant link's content appears under "### Link: <type>"

    Args:
        url (str): The URL of the landing page to fetch.

    Returns:
        str: A formatted string containing:
            - The landing page content
            - The content of each relevant linked page

    Raises:
        None explicitly. Any request-related errors from fetching linked pages
        are caught and skipped. Errors from fetching the main page may propagate
        depending on the implementation of `fetch_website_contents`.

    Side Effects:
        - Prints progress messages to stdout.
        - Makes network requests to the provided URL and its relevant links.

    Dependencies:
        - fetch_website_contents(url): Function that retrieves and returns
          the textual content of a webpage.
        - select_relevant_links(url): Function that returns a list of dicts,
          each containing:
              {
                  "url": <str>,   # लिंक URL
                  "type": <str>   # label/category (e.g., "about", "contact")
              }

    Example:
        >>> result = fetch_page_and_all_relevant_links("https://example.com")
        >>> print(result)

        ## Landing Page:
        <main page content>

        ## Relevant Links:

        ### Link: about
        <about page content>

        ### Link: contact
        <contact page content>

    Notes:
        - The number and quality of fetched links depend on
          `select_relevant_links`.
        - No deduplication or concurrency is implemented.
        - Large pages or many links may result in a very large output string.
    """

    print("Started fetch pages...")
    contents = fetch_website_contents(url)
    print("-"*60)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links:
        print("The link: ", link)
        page = link["url"]
        try:
            fetched = fetch_website_contents(page)
            result += f"\n\n### Link: {link['type']}\n"
            result += fetched
        except requests.exceptions.RequestException as e:
            print(f"Skipping {page}: {e}")    
    return result

In [61]:
display(Markdown(fetch_page_and_all_relevant_links("https://huggingface.co")))

Started fetch pages...
------------------------------------------------------------
Selecting relevant links for https://huggingface.co by calling llama3.2:1b
------------------------------------------------------------
link_system_prompt:
 
* You are provided with a list of links found on a webpage.
* You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Services/Products pages, or Blog/News pages, or Careers/Jobs pages.
* The pages cold be on other languages special portuguese, or spanish
* You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "Jobs page", "url": "https://another.full.url/careers"},
        {"type": "Blog page", "url": "https://another.full.url/blog"},
        {"type": "Products page", "url": "https://another.full.url/products"},
        {"type": "Services pa

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.6-35B-A3B
Updated
about 13 hours ago
•
583k
•
1.24k
moonshotai/Kimi-K2.6
Updated
2 days ago
•
54.5k
•
807
unsloth/Qwen3.6-35B-A3B-GGUF
Updated
2 days ago
•
1.11M
•
667
tencent/HY-World-2.0
Updated
about 14 hours ago
•
548
Qwen/Qwen3.6-27B
Updated
about 10 hours ago
•
390
Browse 2M+ models
Spaces
Running
on
Zero
MCP
2.3k
Wan2.2 14B Preview
🐌
2.3k
generate a video from an image with a text prompt
Running
on
Zero
Agents
Featured
660
OmniVoice
🌍
660
High-quality voice cloning TTS for 600+ languages
Running
156
Bonsai 1-bit WebGPU
🌳
156
Run 1-bit Bonsai LLMs locally in your browser on WebGPU
Running
on
Zero
MCP
Featured
960
FireRed Image Edit 1.0 Fast
🌖
960
FireRed-Image-Edit × Qwen-Image-Edit-Rapid (Transformers)
Running
on
CPU Upgrade
89
ML Intern
🤖
89
Chat with an AI “ML Intern” to get instant ML help
Browse 1M+ applications
Datasets
lambda/hermes-agent-reasoning-traces
Updated
6 days ago
•
7.29k
•
217
Roman1111111/claude-opus-4.6-10000x
Updated
17 days ago
•
6.65k
•
262
Jackrong/GLM-5.1-Reasoning-1M-Cleaned
Updated
4 days ago
•
1.3k
•
55
Kassadin88/GLM-5.1-1000000x
Updated
6 days ago
•
923
•
38
ShadenA/MathNet
Updated
1 day ago
•
1.72k
•
35
Browse 500k+ datasets
The Home of Machine Learning
Create, discover and collaborate on ML better.
The collaboration platform
Host and collaborate on unlimited public models, datasets and applications.
Move faster
With the HF Open source stack.
Explore all modalities
Text, image, video, audio or even 3D.
Build your portfolio
Share your work with the world and build your ML profile.
Sign Up
Accelerate your ML
We provide paid Compute and Enterprise solutions.
Team & Enterprise
Give 
## Relevant Links:


### Link: About page
about (Sergei)

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
Sergei
about
Follow
selvivincent's profile picture
jondelamothe's profile picture
AlbertRuan's profile picture
5
					followers
·
0 following
AI & ML interests
None yet
Organizations
None yet
models
0
None public yet
datasets
0
None public yet
System theme
Company
TOS
Privacy
About
Careers
Website
Models
Datasets
Spaces
Pricing
Docs

### Link: LinkedIn company page
Hugging Face | LinkedIn

Skip to main content
LinkedIn
Top Content
People
Learning
Jobs
Games
Get the app
Sign in
Join now
Hugging Face
Software Development
The AI community building the future.
See jobs
Follow
Discover all 725 employees
Report this company
About us
The AI community building the future.
Website
https://huggingface.co
External link for Hugging Face
Industry
Software Development
Company size
51-200 employees
Type
Privately Held
Founded
2016
Specialties
machine learning, natural language processing, and deep learning
Products
Hugging Face
Hugging Face
Natural Language Processing (NLP) Software
We’re on a journey to solve and democratize artificial intelligence through natural language.
Locations
Primary
Get directions
Paris, FR
Get directions
Employees at Hugging Face
Ludovic Huraux
Rajat Arya
Jeff Boudier
Terrence Rohan
See all employees
Updates
Hugging Face
reposted this
Sayak Paul
12h
Edited
Report this post
My presentation at the
PyTorch
Conference EU 2026 is now live 🔥

I covered how `torch.compile` integrates into Diffusers across different use cases, such as offloading and LoRA hotswapping.

I presented actual numbers along with the other trade-offs involved in the mix, such as compilation time and memory consumption.

I can promise that if you're serious about making `torch.compile` work the right way, you won't be disappointed with the materials presented.

This was quite a bit of teamwork, and I want to thank
Animesh Jain
and
Benjamin Bossan
for the awesome collaboration!

Slides and the recording links are in the comments ⬇️
136
6 Comments
Like
Comment
Share
Hugging Face
reposted this
Daniel van Strien
1d
Report this post
Agents are getting good at helping with a task that used to require a human with headphones: finding the interesting moments in hours of audio.

You can point an agent at a URL and have it help you curate any audio archive — podcast back-catalogues, recorded interviews, lecture series — end-to-end on
Hugging Face
Buckets an

### Link: Twitter handle
No title found

JavaScript is not available.
We’ve detected that JavaScript is disabled in this browser. Please enable JavaScript or switch to a supported browser to continue using x.com. You can see a list of supported browsers in our Help Center.
Help Center
Terms of Service
Privacy Policy
Cookie Policy
Imprint
Ads info
© 2026 X Corp.
Something went wrong, but don’t fret — let’s give it another shot.
Try again
Some privacy related extensions may cause issues on x.com. Please disable them and try again.

### Link: GitHub repository
Hugging Face · GitHub

Skip to content
Navigation Menu
Toggle navigation
Sign in
Appearance settings
huggingface
Platform
AI CODE CREATION
GitHub Copilot
Write better code with AI
GitHub Spark
Build and deploy intelligent apps
GitHub Models
Manage and compare prompts
MCP Registry
New
Integrate external tools
DEVELOPER WORKFLOWS
Actions
Automate any workflow
Codespaces
Instant dev environments
Issues
Plan and track work
Code Review
Manage code changes
APPLICATION SECURITY
GitHub Advanced Security
Find and fix vulnerabilities
Code security
Secure your code as you build
Secret protection
Stop leaks before they start
EXPLORE
Why GitHub
Documentation
Blog
Changelog
Marketplace
View all features
Solutions
BY COMPANY SIZE
Enterprises
Small and medium teams
Startups
Nonprofits
BY USE CASE
App Modernization
DevSecOps
DevOps
CI/CD
View all use cases
BY INDUSTRY
Healthcare
Financial services
Manufacturing
Government
View all industries
View all solutions
Resources
EXPLORE BY TOPIC
AI
Software Development
DevOps
Security
View all topics
EXPLORE BY TYPE
Customer stories
Events & webinars
Ebooks & reports
Business insights
GitHub Skills
SUPPORT & SERVICES
Documentation
Customer support
Community forum
Trust center
Partners
View all resources
Open Source
COMMUNITY
GitHub Sponsors
Fund open source developers
PROGRAMS
Security Lab
Maintainer Community
Accelerator
GitHub Stars
Archive Program
REPOSITORIES
Topics
Trending
Collections
Enterprise
ENTERPRISE SOLUTIONS
Enterprise platform
AI-powered developer platform
AVAILABLE ADD-ONS
GitHub Advanced Security
Enterprise-grade security features
Copilot for Business
Enterprise-grade AI features
Premium Support
Enterprise-grade 24/7 support
Pricing
Search or jump to...
Search code, repositories, users, issues, pull requests...
Search
Clear
Search syntax tips
Provide feedback
We read every piece of feedback, and take your input very seriously.
Include my email address so I can be contacted
Cancel
Submit feedback
Saved searches
Use saved sear

In [16]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [9]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [55]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Started fetch pages...
------------------------------------------------------------
Selecting relevant links for https://huggingface.co by calling llama3.2:1b
------------------------------------------------------------
link_system_prompt:
 
* You are provided with a list of links found on a webpage.
* You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Services/Products pages, or Blog/News pages, or Careers/Jobs pages.
* The pages cold be on other languages special portuguese, or spanish
* You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "Jobs page", "url": "https://another.full.url/careers"},
        {"type": "Blog page", "url": "https://another.full.url/blog"},
        {"type": "Products page", "url": "https://another.full.url/products"},
        {"type": "Services pa

"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.6-35B-A3B\nUpdated\nabout 13 hours ago\n•\n583k\n•\n1.23k\nmoonshotai/Kimi-K2.6\nUpdated\n2 days ago\n•\n54.5k\n•\n807\nunsloth/Qwen3.6-35B-A3B-GGUF\nUpdated\n2 days ago\n•\n1.11M\n•\n667\ntencent/HY-World-2.0\nUpdated\nabout 14 hours ago\n•\n548\nQwen/Qwen3.6-27B\nUpdated\nabout 10 hours ago\n•\n389\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n2.3k\nWan2.2 14B Preview\

In [11]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [59]:
create_brochure("HuggingFace", "https://huggingface.co")

Started fetch pages...
------------------------------------------------------------
Selecting relevant links for https://huggingface.co by calling llama3.2:1b
------------------------------------------------------------
link_system_prompt:
 
* You are provided with a list of links found on a webpage.
* You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Services/Products pages, or Blog/News pages, or Careers/Jobs pages.
* The pages cold be on other languages special portuguese, or spanish
* You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "Jobs page", "url": "https://another.full.url/careers"},
        {"type": "Blog page", "url": "https://another.full.url/blog"},
        {"type": "Products page", "url": "https://another.full.url/products"},
        {"type": "Services pa

# About Hugging Face

## Welcome to Hugging Face

At Hugging Face, we're dedicated to building the future of machine learning and making it accessible to everyone. Our mission is to empower the AI community by providing a comprehensive platform for model development, deployment, and collaboration.

## Our Story

Founded in 2016, Hugging Face has grown into one of the most influential players in the machine learning ecosystem. Our team consists of passionate individuals from around the world, united by our shared goal of advancing artificial intelligence through natural language processing (NLP) and deep learning.

## Our Mission

Hugging Face is on a journey to solve and democratize artificial intelligence through natural language. We believe that everyone should have access to high-quality AI models and tools to build impactful applications. By providing a user-friendly platform, we aim to bridge the gap between research and production, making it easier for developers and researchers to create innovative AI solutions.

## Our Values

*   **Open-Source**: We believe in transparency and open-source principles, sharing our knowledge and code with the community.
*   **Community-Driven**: Our strength lies in our collaborative environment, where we work together to develop and improve our platform.
*   **Innovation**: Always pushing the boundaries of what's possible with AI, we innovate and adopt cutting-edge technologies to stay ahead.

## What We Do

Our platform offers a suite of tools and services designed for model development, deployment, and collaboration:

*   **Hugging Face Hub**: A centralized marketplace for models, datasets, and applications.
*   **Inference Endpoints**: Easy deployment of AI models to production with full managed infrastructure.
*   **Datasets**: A vast collection of pre-processed and high-quality datasets for training and testing.

## Our Impact

With Hugging Face, you're not just using a tool – you're joining a community that's shaping the future of artificial intelligence. Our platform has democratized access to AI, empowering developers and researchers around the world.

## Join Us

Ready to be part of this journey? Explore our [careers page](#careers) or learn more about our [community](#community). Together, we can build a brighter future for everyone.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [12]:
def stream_brochure(company_name, url, langage: str = "English", tone: str = None):
    system_prompt = brochure_system_prompt + f"\nIn {langage}"
    if tone:
        system_prompt += f"\nWith a {tone} tone"
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("Vitarts", "http://vitarts.com.br", "Brazilian Portuguese")

Started fetch pages...
------------------------------------------------------------
Selecting relevant links for http://vitarts.com.br by calling llama3.2:1b
------------------------------------------------------------
link_system_prompt:
 
* You are provided with a list of links found on a webpage.
* You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Services/Products pages, or Blog/News pages, or Careers/Jobs pages.
* The pages cold be on other languages special portuguese, or spanish
* You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "Jobs page", "url": "https://another.full.url/careers"},
        {"type": "Blog page", "url": "https://another.full.url/blog"},
        {"type": "Products page", "url": "https://another.full.url/products"},
        {"type": "Services pag

# Vitarts: Transformando Negócios com Tecnologia e Gestão

**Sobre Nós**

A Vitarts é uma startup de consultoria especializada em tecnologias de gestão e inteligência de negócios. Com sede no Rio de Janeiro, nossa empresa atua em todo o território nacional, oferecendo soluções integradas a clientes que buscam elevar sua eficiência e eficácia.

**Nossa Missão**

Apoiar nossos clientes a elevar sua eficiência e eficácia na melhoria continua de seus processos através de serviços e soluções integradas de gestão e inteligência de negócio.

**Nossa Visão**

Ser referência para serviços e soluções que transformam os negócios e operações de nossos clientes através de formas mais ágeis, simples e econômicas.

**Nossos Valores**

*   Inovação: Atuar com criatividade, inteligência e talento para identificar e aplicar novas formas e soluções objetivando a melhoria continua e captura de oportunidades;
*   Sustentabilidade: Ser responsável na condução de nossas operações, com foco no crescimento sustentável e o atingimento dos objetivos de nossos negócios e de nossos clientes;
*   Ética: Primar pela integridade de nossa reputação, agindo conscientemente e responsavelmente em todas as nossas ações.

**Nossos Serviços**

A Vitarts oferece uma gama completa de serviços, incluindo:

*   Soluções integradas de gestão e inteligência de negócio;
*   Consultoria especializada em tecnologias de gestão;
*   Suporte e manutenção de sistemas;

**Nossa Equipe**

Nossa equipe é formada por profissionais experientes e talentosos, altamente qualificados para fornecer os melhores serviços ao nosso cliente. Nossa equipe é liderada pelo Marcelo Marques, que tem uma forte experiência no campo da consultoria em gestão e tecnologia.

**Estamos Aqui Para**

Ajudar o seu negócio a decolar! Entre em contato conosco para saber mais sobre como podemos ajudá-lo.

In [17]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co", "humorous, use emojes and give a irreverence")

Started fetch pages...
------------------------------------------------------------
Selecting relevant links for https://huggingface.co by calling llama3.2:1b
------------------------------------------------------------
link_system_prompt:
 
* You are provided with a list of links found on a webpage.
* You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Services/Products pages, or Blog/News pages, or Careers/Jobs pages.
* The pages cold be on other languages special portuguese, or spanish
* You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "Jobs page", "url": "https://another.full.url/careers"},
        {"type": "Blog page", "url": "https://another.full.url/blog"},
        {"type": "Products page", "url": "https://another.full.url/products"},
        {"type": "Services pa

**The Hugging Face Brochure**

[Cover Image: A playful illustration of a group of developers huddled around a computer, surrounded by AI-powered robots and code snippets]

Welcome to Hugging Face, the AI community that's changing the game!

**Unlocking Human-AI Collaboration**

Hugging Face is more than just an AI company – we're building a platform where humans and machines collaborate to create innovative solutions. We empower developers, researchers, and businesses to drive breakthroughs in machine learning with our cutting-edge tools and resources.

**Our Community**

We have a vibrant community of over 2 Million+ ML models, 500k+ datasets, and thousands of applications. Our collaborations are open, public, and accessible, ensuring that everyone can contribute and learn from each other's work.

[Image: A graphic showcasing the diversity of our users, with icons representing different job roles (researcher, developer, entrepreneur), age groups, locations]

**Who We Help**

We serve individuals, researchers, businesses, and organizations worldwide. Whether you're just starting out with ML or building a scalable enterprise solution, we've got your back.

[Image: A simple illustration of a person using a computer, surrounded by thought bubbles containing AI-powered ideas]

**What We Offer**

* **Unlimited Public Models**: Harness our 2 Million+ models and create innovative solutions
* **Datasests & Applications**: Leverage our 500k+ datasets and build applications for the modern world
* **AI-powered Collaboration Platform**: Join our vibrant community, upload your work, and get instant ML feedback!
* **Compute & Enterprise Solutions**: Get scalable compute resources and expert support tailored to your business needs

**Benefits of Working with Hugging Face**

Get access to cutting-edge AI tools, collaborative platforms, and global expertise. Whether you're looking for inspiration, resources, or expert guidance, we've got the Hugging Face community working for you!

How does it work?

[Image: A simplified illustration of a user signing up, connecting their account, and accessing the platform]

1. **Sign-up**: Create an account and start exploring our platform
2. **Join Our Community**: Publish your models, datasets, or applications, and collaborate with others
3. **Expert Support**: Get instant help from our AI-powered chatbot or expert support team

[Cover Image: A group of diverse developers huddled around a computer, surrounded by AI-powered robots and code snippets]

Ready to unlock human-AI collaboration?

Apply now, join our community, and get ready to change the world with machine learning!

Best regards,
The Hugging Face Team

In [82]:
stream_brochure("Vitarts", "http://vitarts.com.br")

Started fetch pages...
------------------------------------------------------------
Selecting relevant links for http://vitarts.com.br by calling llama3.2:1b
------------------------------------------------------------
link_system_prompt:
 
* You are provided with a list of links found on a webpage.
* You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Services/Products pages, or Blog/News pages, or Careers/Jobs pages.
* The pages cold be on other languages special portuguese, or spanish
* You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "Jobs page", "url": "https://another.full.url/careers"},
        {"type": "Blog page", "url": "https://another.full.url/blog"},
        {"type": "Products page", "url": "https://another.full.url/products"},
        {"type": "Services pag

# Vitarts: A Partner for Your Business Growth

At Vitarts, we believe in turning data into actionable insights to drive business success. As a specialized IT consulting company, our expertise extends from strategic planning to implementation and operation of integrated solutions.

### Our Story

Our team of experienced professionals has worked together since 2017, bringing together creative ideas, innovative thinking, and sheer determination. We have the vision to help businesses like yours take off. With a proven track record in delivering results-driven solutions, we are your trusted advisor for any IT-related challenge.

### Services Offered

* Strategic Planning
* Integrated Solutions Implementation
* Data Analytics and Business Intelligence
* Marketing Research and Analysis
* Sales Dashboard by Region
* Operation and Maintenance of Integrated Systems

### Aiming for Results-Oriented Success

We employ an agile methodology, focusing on measurable outcomes, to deliver value to our clients. This approach ensures that every project meets the client's needs and satisfies their expectations.

### Testimonials from Clients

Marcelo is a highly skilled consultant with technical capabilities in Business Intelligence. He is an excellent communicator who effectively bridges the gap between internal stakeholders and clients.
 
A business-oriented individual like Marcelo combines his exceptional technical expertise with strong analytical skills and an eye for details, ensuring success in both personal and professional endeavors.

We are proud to offer our expertise as a collaborative partner, leveraging the results-driven approach we take in every project. We believe this unique blend of technical capabilities, strategic planning, and innovative thinking is essential for companies seeking to excel in their business objectives.


### Join Our Team

If you're interested in joining our dynamic team of innovators and industry experts, check out the Opportunities section for available openings.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>